In [87]:
import pandas as pd
import os
import glob

# Specify the path to your folder containing CSV files
folder_path = 'StudentProficiencyData'

# Use glob to get a list of all CSV files in the folder
file_pattern = os.path.join(folder_path, '*.csv.gz')
file_list = glob.glob(file_pattern)

# Initialize an empty DataFrame to concatenate the individual files
df_list = []

column_names = ['studentid', 'previousitems', 'itemdifficulty', 'currentperformance', 'currentscore', 'itemid', 'eventtime', 'skillid', 'subskillid']

# Loop through each file and load it into a DataFrame with the specified column names and data types
for file in file_list:
    df = pd.read_csv(file, compression='gzip', names=column_names, parse_dates=['eventtime'], na_values=['NA'])
    
    # Explicitly convert 'previousitems' column to integer
    #df['previousitems'] = pd.to_numeric(df['previousitems'], errors='coerce', downcast='integer')
    #df['currentscore'] = pd.to_numeric(df['currentscore'], errors='coerce', downcast='float')
    
    df_list.append(df)
    
# Concatenate all DataFrames in the list into a single DataFrame
final_df = pd.concat(df_list, ignore_index=True)

# Now, final_df contains the data from all CSV files


In [88]:
final_df

,studentid,previousitems,itemdifficulty,currentperformance,currentscore,itemid,eventtime,skillid,subskillid
0,26589996,2,0.499486,100,0.656238,3082571b426f,2023-12-05 14:39:42.334,d5a4bfaa479de311b77c005056801da1,ootainee
1,26589996,2,0.228293,100,1.128271,3ad45b088b9e,2023-12-05 14:39:42.339,d5a4bfaa479de311b77c005056801da1,phahnoth
2,26589996,2,-0.680715,0,-0.075620,3bf979d1519a,2023-12-05 14:39:42.344,d5a4bfaa479de311b77c005056801da1,dohweewo
3,40085150,3,-0.105964,100,0.500231,f6166b65e327,2023-12-05 14:39:42.348,72a3bfaa479de311b77c005056801da1,\\N
4,40085150,9,-0.285237,100,0.644498,fd59caa59c3e,2023-12-05 14:39:42.353,70a3bfaa479de311b77c005056801da1,\\N
...,...,...,...,...,...,...,...,...,...
21436056,42007008,2,0.498613,100,1.185687,694f17199345,2023-12-05 15:19:36.053,99a4bfaa479de311b77c005056801da1,\\N
21436057,37284623,2,0.316429,100,1.592607,9301b2dd4532,2023-12-05 15:19:36.057,45a3bfaa479de311b77c005056801da1,\\N
21436058,37284623,3,0.219354,100,1.291225,a5c2abd9a613,2023-12-05 15:19:36.062,45a3bfaa479de311b77c005056801da1,\\N
21436059,37284623,4,0.066582,0,0.611089,9d9d84fc8061,2023-12-05 15:19:36.068,50a3bfaa479de311b77c005056801da1,45a3bfaa479de311b77c005056801da1


In [14]:
!pip install surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.0/772.0 kB 35.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.3-cp310-cp310-linux_x86_64.whl size=1381417 sha256=393e93f9920273eac91023980c1217dc66bb566b8e974b86023dbbfac05d0d3a
  Stored in directory: /home/ec2-user/.cache/pip/wheels/a5/ca/a8/4e28def53797fdc4363ca4af740db15a9c2f1595ebc51fb445
Successfully built scikit-surprise


In [89]:
# Keep only the latest entry for each skillid
latest_scores = final_df.groupby(['studentid', 'skillid']).last().reset_index()
latest_scores

,studentid,skillid,previousitems,itemdifficulty,currentperformance,currentscore,itemid,eventtime,subskillid
0,1707368,37bdc51aa39fe3119503005056801da1,25,0.345682,100,1.118518,49e1e7a433d7,2023-12-06 16:47:30.788,\\N
1,1707368,68a4bfaa479de311b77c005056801da1,7,0.234552,100,0.955899,84ce0de9cdc0,2023-12-06 16:44:09.243,\\N
2,1707368,69a4bfaa479de311b77c005056801da1,5,0.002732,0,1.044169,7a7e065f2878,2023-12-06 16:40:15.724,\\N
3,1707368,6ba4bfaa479de311b77c005056801da1,4,-0.270160,0,0.280686,6981ccfdb38a,2023-12-08 15:07:14.719,\\N
4,1707368,6fa4bfaa479de311b77c005056801da1,20,0.167560,100,0.751880,0f8f953d8988,2023-12-08 15:15:10.461,\\N
...,...,...,...,...,...,...,...,...,...
4064571,47438608,afa4bfaa479de311b77c005056801da1,2,-0.329972,0,-0.129089,3bda9e5856b1,2023-12-19 21:15:40.973,\\N
4064572,47438608,ahvohcoh,2,-0.882960,0,-0.009551,b4fa40b1f060,2023-12-19 21:18:57.633,\\N
4064573,47438608,d0a3bfaa479de311b77c005056801da1,2,-0.318862,100,-0.019928,b761b0973165,2023-12-19 21:28:22.609,\\N
4064574,47438620,99a4bfaa479de311b77c005056801da1,2,0.453962,100,0.992992,8e06887a28a1,2023-12-19 21:27:53.854,\\N


In [ ]:
from surprise import Dataset, Reader, SVD
import pandas as pd

# Assuming the dataset is in a Pandas DataFrame with columns ['studentid', 'skillid', 'currentscore']
reader = Reader(rating_scale=(0, 1))

# Load the data into a Surprise Dataset
data = Dataset.load_from_df(latest_scores[['studentid', 'skillid', 'currentscore']], reader)


# Assuming data is your Surprise Dataset
unique_user_ids = latest_scores['studentid'].unique()

# Split the user IDs into training and testing sets
split_percentage = 0.8
split_index = int(len(unique_user_ids) * split_percentage)
train_user_ids, test_user_ids  = np.split(unique_user_ids, [split_index])
#train_user_ids, test_user_ids = train_test_split(unique_user_ids, test_size=0.2, random_state=42)

# Construct the training set
trainset = data.build_full_trainset()
trainset_for_test = trainset.build_testset()
trainset_for_test = [elem for elem in trainset_for_test if elem[0] in train_user_ids]

# Use SVD algorithm
algorithm = SVD()

# Train the model
trainset = data.build_full_trainset()
algorithm.fit(trainset)


In [ ]:
# Construct the test set
testset = trainset.build_testset()
testset = [elem for elem in testset if elem[0] in test_user_ids]

# Make predictions on the test set
predictions = algorithm.test(testset)
predictions


In [104]:

# Compute RMSE and MAE for the entire test set
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

# Print the results for the entire test set
print(f'RMSE (Entire Test Set): {rmse}')
print(f'MAE (Entire Test Set): {mae}')

RMSE: 0.3126
MAE:  0.2478
RMSE (Entire Test Set): 0.3126105418877624
MAE (Entire Test Set): 0.24781477832924237


In [105]:
# Hold back one skill ID during prediction for evaluation
unique_skill_ids = latest_scores['skillid'].unique()

for held_back_skill_id in unique_skill_ids:
    # Filter predictions and ground truth for the held-back skill ID
    skill_predictions = [pred for pred in predictions if pred[1] == held_back_skill_id]
    skill_ground_truth = [(elem[0], elem[1], elem[2]) for elem in testset if elem[1] == held_back_skill_id]

    # Check if there are predictions and ground truth for the skill ID
    if skill_predictions and skill_ground_truth:
        # Compute RMSE and MAE for the held-back skill ID
        rmse_skill = accuracy.rmse(skill_predictions)
        mae_skill = accuracy.mae(skill_predictions)

        # Print the results for the held-back skill ID
        print(f'\nHeld Back Skill ID: {held_back_skill_id}')
        print(f'RMSE (Held Back Skill ID): {rmse_skill}')
        print(f'MAE (Held Back Skill ID): {mae_skill}')

RMSE: 0.2657
MAE:  0.2093

Held Back Skill ID: 37bdc51aa39fe3119503005056801da1
RMSE (Held Back Skill ID): 0.26568498260896806
MAE (Held Back Skill ID): 0.2093042204632251
RMSE: 0.3369
MAE:  0.2732

Held Back Skill ID: 68a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.336914968265993
MAE (Held Back Skill ID): 0.27319473844277253
RMSE: 0.3474
MAE:  0.2838

Held Back Skill ID: 69a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.3474443774086678
MAE (Held Back Skill ID): 0.2838407725348209
RMSE: 0.2716
MAE:  0.2128

Held Back Skill ID: 6ba4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.27161775748256867
MAE (Held Back Skill ID): 0.21275923376050387
RMSE: 0.3271
MAE:  0.2748

Held Back Skill ID: 6fa4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.3270800162239255
MAE (Held Back Skill ID): 0.2748082197670019
RMSE: 0.3470
MAE:  0.2892

Held Back Skill ID: 78a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.34701386745754814
MAE (Held Back 

RMSE: 0.2803
MAE:  0.2316

Held Back Skill ID: 84a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.28028269906681347
MAE (Held Back Skill ID): 0.2315878227124722
RMSE: 0.3215
MAE:  0.2642

Held Back Skill ID: 87a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.321544784919998
MAE (Held Back Skill ID): 0.2641867797586055
RMSE: 0.3070
MAE:  0.2578

Held Back Skill ID: 735f12e14b3d42eb86dc02816aba0cf8
RMSE (Held Back Skill ID): 0.30696893563631306
MAE (Held Back Skill ID): 0.25777349920801773
RMSE: 0.3212
MAE:  0.2532

Held Back Skill ID: 06a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.3212208711694891
MAE (Held Back Skill ID): 0.25319494478081583
RMSE: 0.3540
MAE:  0.2829

Held Back Skill ID: 1da5bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.3539583909766342
MAE (Held Back Skill ID): 0.28288358382375894
RMSE: 0.3375
MAE:  0.2839

Held Back Skill ID: 20a5bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.33745805820936464
MAE (Held Back

RMSE: 0.3184
MAE:  0.2611

Held Back Skill ID: d7a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.31841913179006265
MAE (Held Back Skill ID): 0.26105281414838705
RMSE: 0.3517
MAE:  0.3001

Held Back Skill ID: dahteeco
RMSE (Held Back Skill ID): 0.35172322824492985
MAE (Held Back Skill ID): 0.3001175914202269
RMSE: 0.3224
MAE:  0.2642

Held Back Skill ID: eca4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.32242960619635674
MAE (Held Back Skill ID): 0.26424163270316875
RMSE: 0.2992
MAE:  0.2408

Held Back Skill ID: kpkxgrti
RMSE (Held Back Skill ID): 0.29920479379009934
MAE (Held Back Skill ID): 0.24080161402012
RMSE: 0.3186
MAE:  0.2527

Held Back Skill ID: 341bf9aee38846df9c59842a7b460015
RMSE (Held Back Skill ID): 0.3186312038224271
MAE (Held Back Skill ID): 0.25270358985667946
RMSE: 0.3231
MAE:  0.2664

Held Back Skill ID: 44a3bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.3231130322191411
MAE (Held Back Skill ID): 0.2663569134581475
RMSE: 0.2135
MAE:

RMSE: 0.3556
MAE:  0.2854

Held Back Skill ID: d9a3bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.35561214742564523
MAE (Held Back Skill ID): 0.28536317484831186
RMSE: 0.3302
MAE:  0.2568

Held Back Skill ID: 02a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.33021625048217773
MAE (Held Back Skill ID): 0.2567904012425813
RMSE: 0.2968
MAE:  0.2370

Held Back Skill ID: 03a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.2968024281505736
MAE (Held Back Skill ID): 0.2369856000942157
RMSE: 0.3280
MAE:  0.2703

Held Back Skill ID: 05a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.3279906631145399
MAE (Held Back Skill ID): 0.27034050738573173
RMSE: 0.3533
MAE:  0.3006

Held Back Skill ID: 07a4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.35329337531600635
MAE (Held Back Skill ID): 0.3006228905568536
RMSE: 0.2579
MAE:  0.2023

Held Back Skill ID: 0da4bfaa479de311b77c005056801da1
RMSE (Held Back Skill ID): 0.25794855311994186
MAE (Held Bac

KeyboardInterrupt: 

In [106]:
held_back_skill_id

'11a4bfaa479de311b77c005056801da1'

In [116]:
#fake test user

#predictions = algorithm.test(testset)

faketest = [(999999999,'eda4bfaa479de311b77c005056801da1',1),
            (999999999,'3ea5bfaa479de311b77c005056801da1',1),
            (999999999,'5fa5bfaa479de311b77c005056801da1',1)]

new_user_skills=[]
for row in faketest:
    new_user_skills.append(row[1])
    
print(new_user_skills)

new_user_predictions = []
for skill_id in unique_skill_ids:
    if skill_id not in new_user_skills:
        # Make a prediction for each unseen skill
        prediction = algorithm.predict('new_user', skill_id)
        new_user_predictions.append(prediction)

for prediction in new_user_predictions:
    print(f"User ID: {prediction.uid}, Skill ID: {prediction.iid}, True Rating: {prediction.r_ui}, Estimated Rating: {prediction.est}")


['eda4bfaa479de311b77c005056801da1', '3ea5bfaa479de311b77c005056801da1', '5fa5bfaa479de311b77c005056801da1']
User ID: new_user, Skill ID: 37bdc51aa39fe3119503005056801da1, True Rating: None, Estimated Rating: 0.7037768839488503
User ID: new_user, Skill ID: 68a4bfaa479de311b77c005056801da1, True Rating: None, Estimated Rating: 0.6605679794474429
User ID: new_user, Skill ID: 69a4bfaa479de311b77c005056801da1, True Rating: None, Estimated Rating: 0.5249970936306955
User ID: new_user, Skill ID: 6ba4bfaa479de311b77c005056801da1, True Rating: None, Estimated Rating: 0.35608884937839397
User ID: new_user, Skill ID: 6fa4bfaa479de311b77c005056801da1, True Rating: None, Estimated Rating: 0.5362811115012548
User ID: new_user, Skill ID: 78a4bfaa479de311b77c005056801da1, True Rating: None, Estimated Rating: 0.4739445256592183
User ID: new_user, Skill ID: 7db6668c36134063a06b6e23e391f5a5, True Rating: None, Estimated Rating: 0.5422012272960044
User ID: new_user, Skill ID: 933af6788dec46fa80ee5af76df7

In [ ]:
# Now, let's evaluate by dropping one skill_id score at a time for each user
# Accumulate true scores and predictions for each modification
true_scores = []
modified_predictions = []

#get all user ids in testset
unique_user_ids = list(set([elem[0] for elem in testset]))

# Iterate over each user in the test set
for user_id in unique_user_ids: 
    user_testset = [elem for elem in testset if elem[0] == user_id]

    for true_skill_id, true_score, _ in user_testset:
        # Copy the user test set and remove the true skill_id score
        modified_testset = [elem for elem in user_testset if elem[1] != true_skill_id]

        # Make a prediction for the modified test set
        modified_prediction = algorithm.predict(user_id, true_skill_id)

        # Accumulate true scores and predictions
        true_scores.append(true_score)
        modified_predictions.append(modified_prediction.est)

# Compute RMSE and MAE for all dropout predictions
rmse_dropout = accuracy.rmse([(true, pred) for true, pred in zip(true_scores, modified_predictions)])
mae_dropout = accuracy.mae([(true, pred) for true, pred in zip(true_scores, modified_predictions)])

print(f'RMSE on Dropout Predictions: {rmse_dropout}')
print(f'MAE on Dropout Predictions: {mae_dropout}')

In [ ]:
rmse_dropout = accuracy.rmse([(true, pred) for true, pred in zip(true_scores, modified_predictions)])
mae_dropout = accuracy.mae([(true, pred) for true, pred in zip(true_scores, modified_predictions)])

print(f'RMSE on Dropout Predictions: {rmse_dropout}')
print(f'MAE on Dropout Predictions: {mae_dropout}')

In [17]:
accuracy.rmse(predictions)


RMSE: 0.2951


0.2950919756397519

In [52]:

# Specify the user for whom you want to impute ratings for all other skills
user_id = 44221404

# Get all skills (items) that the user has not rated
all_skills = final_df['skillid'].unique()
rated_skills = final_df[final_df['studentid'] == user_id]['skillid'].unique()
unrated_skills = set(all_skills) - set(rated_skills)

# Generate predictions for unrated skills
imputed_ratings = [(user_id, item_id, algorithm.predict(user_id, item_id).est) for item_id in unrated_skills]

# Print the imputed ratings
for user_id, item_id, imputed_rating in imputed_ratings:
    print(f"User {user_id} - Skill {item_id}: Imputed Rating = {imputed_rating}")

User 44221404 - Skill 9f04bd21acc34c6f849324d339c84422: Imputed Rating = 0.859542726720917
User 44221404 - Skill 697e63d82400e4119e54005056801da1: Imputed Rating = 0.6166621736229205
User 44221404 - Skill a6a3bfaa479de311b77c005056801da1: Imputed Rating = 0.6670623806561423
User 44221404 - Skill 27a3bfaa479de311b77c005056801da1: Imputed Rating = 0.5963353047297132
User 44221404 - Skill 31bec51aa39fe3119503005056801da1: Imputed Rating = 0.629265221579426
User 44221404 - Skill 28a6bfaa479de311b77c005056801da1: Imputed Rating = 0.5347882231166186
User 44221404 - Skill ooghahte: Imputed Rating = 0.7395411951665554
User 44221404 - Skill 6ba4bfaa479de311b77c005056801da1: Imputed Rating = 0.4939535673430723
User 44221404 - Skill ada3bfaa479de311b77c005056801da1: Imputed Rating = 0.778126050064583
User 44221404 - Skill bdba0954428c45f99ac6c3f437c645e6: Imputed Rating = 0.8383085107994892
User 44221404 - Skill wifieyei: Imputed Rating = 0.8015341950737063
User 44221404 - Skill d5a4bfaa479de311b

In [44]:
testset

[(44221404, 'aea3bfaa479de311b77c005056801da1', 0.69336673),
 (39485168, '4aa3bfaa479de311b77c005056801da1', 0.85490263),
 (44230524, '10a4bfaa479de311b77c005056801da1', 0.84845094),
 (39888374, '72a3bfaa479de311b77c005056801da1', 0.97266273),
 (42153241, '70f8d27f8b8f4b6083f4b66fbdb10609', 0.88294154),
 (44864367, 'phaofees', 0.1948475),
 (28387154, 'ee30a13bf08e4880bb4a4ec350837440', 0.59155333),
 (46661482, '5ea3bfaa479de311b77c005056801da1', 0.78478662),
 (31719114, '02a5bfaa479de311b77c005056801da1', 0.48460765),
 (34487757, '6d2bd40241ea4dafb74370433442f1bd', 0.3994852),
 (43681938, 'd3a4bfaa479de311b77c005056801da1', 0.67686767),
 (39905349, 'b7e3216a1040491ebbb37d0aa36407a0', 0.9087594),
 (36126618, '94a5bfaa479de311b77c005056801da1', 0.88590755),
 (35865543, '49a3bfaa479de311b77c005056801da1', 0.983710795),
 (45819022, '9fa3bfaa479de311b77c005056801da1', 0.72504506),
 (31284282, 'b975030e84c74d96a51d4df1fad93ff5', 0.96426828),
 (31722420, '21a4bfaa479de311b77c005056801da1', 0.